In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.14 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Co₂SiO₄ on D20 at 2.3 K

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit the magnetic structure of the olivine Co₂SiO₄ to constant-wavelength neutron powder data
from D20 at ILL, measured at 2.3 K, below its ordering temperature. The cobalt moments order with
k = 0 in the magnetic space group Pnma.1: Co1 may point in any direction, Co2 only along b.

It builds the catalogue case `pd-neut-cwl_cosio-d20_2k` step by step: the same project the
command line fits and the app's Examples open. Its two Co sites share one displacement, and the
two FCJ asymmetry terms are tied.

## Import the library

In [2]:
import easydiffraction as edi

## Define the project

A new project is empty: it holds no structure and no experiment until you add them.

In [3]:
project = edi.Project(name='pd-neut-cwl_cosio-d20_2k')

## Define the structure

Load the structure from EasyDiffraction's collection. Its CIF gives the space group, the cell and
the atom sites.

In [4]:
path = edi.download_data('pd-neut-cwl_cosio-d20_2k-cosio', destination='data')
project.structures.add_from_cif_path(path)
structure = project.structures['cosio']

Data 'pd-neut-cwl_cosio-d20_2k-cosio' is at data/pd-neut-cwl_cosio-d20_2k-cosio.edi


In [5]:
project.display.structure('cosio')

View(html=<324,456 characters>, height=660)

## Define the experiment

Create the experiment from the measured data in EasyDiffraction's collection, then describe the
instrument, the peak profile, the background and the structure it links.

In [6]:
data_path = edi.download_data('pd-neut-cwl_cosio-d20_2k-d20-data', destination='data')
project.experiments.add_from_data_path(
    name='d20',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment = project.experiments['d20']
experiment.peak.type = 'cwl-tch-pseudo-voigt-fcj'
experiment.absorption.type = 'cylinder-hewat'
experiment.instrument.name = 'D20'
experiment.instrument.facility = 'ILL'
experiment.peak.broad_gauss_u = 0.273527
experiment.peak.broad_gauss_v = -0.589718
experiment.peak.broad_gauss_w = 0.413007
experiment.peak.broad_lorentz_y = 0.017548
experiment.peak.cutoff_fwhm = 8
experiment.absorption.mu_r = 0.230
experiment.instrument.setup_wavelength = 1.87
experiment.instrument.calib_twotheta_offset = 0.29433
experiment.excluded_regions.create(start=0, end=6)
experiment.excluded_regions.create(start=64.65, end=64.74)
experiment.excluded_regions.create(start=150, end=180)
experiment.linked_structures.create(structure_id='cosio', scale=1.817226)
experiment.background.create(position=6, intensity=603.5)
experiment.background.create(position=6.5, intensity=585.0)
experiment.background.create(position=7, intensity=568.5)
experiment.background.create(position=7.6, intensity=528.8)
experiment.background.create(position=12.6, intensity=418.9)
experiment.background.create(position=13, intensity=420.2)
experiment.background.create(position=13.5, intensity=416.5)
experiment.background.create(position=15, intensity=416.0542)
experiment.background.create(position=25, intensity=394.7747)
experiment.background.create(position=30, intensity=373.1628)
experiment.background.create(position=50, intensity=406.3737)
experiment.background.create(position=70, intensity=408.5056)
experiment.background.create(position=90, intensity=395.9971)
experiment.background.create(position=110, intensity=345.9223)
experiment.background.create(position=130, intensity=279.0409)
experiment.background.create(position=150, intensity=230.6987)

Data 'pd-neut-cwl_cosio-d20_2k-d20-data' is at data/pd-neut-cwl_cosio-d20_2k-d20.xye


The constraints the fit keeps, each between two named parameters (aliases).

In [7]:
project.analysis.aliases.create(id='biso_Co1', param=structure.atom_sites['Co1'].adp_iso)
project.analysis.aliases.create(id='biso_Co2', param=structure.atom_sites['Co2'].adp_iso)
project.analysis.aliases.create(id='sl', param=experiment.peak.asym_fcj_1)
project.analysis.aliases.create(id='dl', param=experiment.peak.asym_fcj_2)
project.analysis.constraints.create(id='biso_Co2', expression='biso_Co2 = biso_Co1')
project.analysis.constraints.create(id='dl', expression='dl = sl')

In [8]:
project.display.constraints()

View(html=<81,033 characters>, height=264)

In [9]:
project.display.pattern('d20')

View(html=<310,964 characters>, height=480)

## Perform the analysis

Everything starts fixed, and five stages each free one more group, keeping the earlier ones free:
the scale, cell and moments, then the background, the atoms, the peak widths and last the peak
asymmetry.

### Scale, 2θ zero, cell and magnetic moments

The first stage frees the phase scale, the 2θ zero shift, the three cell lengths and the magnetic
model: the moment components the magnetic space group allows (Co1 in any direction, Co2 only
along b) and the j2 term of the Co²⁺ form factor, which sets how the magnetic intensities fall
with angle. They are freed together because at 2 K many peaks are purely magnetic: the scale
fitted alone would also absorb the error of the starting moments in those peaks, while freed
with the moments it follows the nuclear peaks and the moments follow the magnetic ones.

In [10]:
experiment.linked_structures['cosio'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True
for parameter in (
    structure.atom_site_moment['Co1'].crystalaxis_x,
    structure.atom_site_moment['Co1'].crystalaxis_y,
    structure.atom_site_moment['Co1'].crystalaxis_z,
    structure.atom_site_moment['Co2'].crystalaxis_y,
):
    parameter.free = True
structure.magnetic_form_factor['Co'].j2_coefficient.free = True

In [11]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,cosio,cell,,length_a,10.3081,0,0,99,Å
2,cosio,cell,,length_b,6.00219,0,0,99,Å
3,cosio,cell,,length_c,4.78739,0,0,99,Å
4,cosio,atom_site_moment,Co1,crystalaxis_x,2,0,-inf,inf,μB
5,cosio,atom_site_moment,Co1,crystalaxis_y,2,0,-inf,inf,μB
6,cosio,atom_site_moment,Co1,crystalaxis_z,2,0,-inf,inf,μB
7,cosio,atom_site_moment,Co2,crystalaxis_y,2,0,-inf,inf,μB
8,cosio,magnetic_form_factor,Co,j2_coefficient,0,0,-inf,inf,
9,d20,instrument,,calib_twotheta_offset,0.29433,0,-inf,inf,deg
10,d20,linked_structure,cosio,scale,1.81723,0,-inf,inf,


In [12]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 10 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       11.38        35.86
      1      0.022      5.97         9.88   72.5% ↓
      2      0.026      5.97         9.88   
  done · converged · 2 iters · reduced χ² 9.88 · Rwp 5.97 % · 0.03 s


View(html=<95,511 characters>, height=376)

In [13]:
project.display.pattern('d20')

View(html=<311,858 characters>, height=480)

### Background

The second stage frees the sixteen background points, which follow the level between the peaks.

In [14]:
for point in experiment.background:
    point.intensity.free = True

In [15]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 26 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.97         9.99
      1      0.029      5.65         8.93   10.6% ↓
  done · converged · 1 iters · reduced χ² 8.93 · Rwp 5.65 % · 0.03 s


View(html=<116,299 characters>, height=376)

In [16]:
project.display.pattern('d20')

View(html=<311,715 characters>, height=480)

### Atomic displacements and positions

The third stage frees the atoms: each site's isotropic displacement (Co2's follows Co1's by the
project's constraint) and the coordinates the symmetry leaves free. Co1 sits on a centre of
symmetry, so it has none; the other sites lie on the mirror plane, except O3. Both are freed in
one stage because positions and displacements change the same peak intensities: fitted apart,
the one freed first would absorb the error of the other.

In [17]:
for site in structure.atom_sites:
    site.adp_iso.free = not site.adp_iso.user_constrained
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [18]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 42 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.65         9.03
      1      0.028      5.59         8.86   1.8% ↓
      2      0.034      5.59         8.86   
  done · converged · 2 iters · reduced χ² 8.86 · Rwp 5.59 % · 0.04 s


View(html=<136,254 characters>, height=376)

In [19]:
project.display.pattern('d20')

View(html=<311,779 characters>, height=480)

### Peak widths

The fourth stage frees the peak profile: the Gaussian widths U, V and W and the Lorentzian Y.

In [20]:
for parameter in (
    experiment.peak.broad_gauss_u,
    experiment.peak.broad_gauss_v,
    experiment.peak.broad_gauss_w,
    experiment.peak.broad_lorentz_y,
):
    parameter.free = True

In [21]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 46 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.59         8.89
      1      0.025      5.58         8.86   0.3% ↓
  done · converged · 1 iters · reduced χ² 8.86 · Rwp 5.58 % · 0.03 s


View(html=<138,756 characters>, height=376)

In [22]:
project.display.pattern('d20')

View(html=<311,885 characters>, height=480)

### Peak asymmetry

The last stage frees the asymmetry: the low-angle peaks lean towards low angle. One FCJ term is
freed; the other follows it by the project's constraint.

In [23]:
experiment.peak.asym_fcj_1.free = True

In [24]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 47 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.58         8.87
      1      0.183      5.51         8.62   2.8% ↓
      2      0.188      5.44         8.41   2.4% ↓
      3      0.193      5.43         8.39   0.2% ↓
      4      0.198      5.36         8.17   2.7% ↓
      5      0.204      5.20         7.69   5.8% ↓
      6      0.209      4.91         6.85   11.0% ↓
      7      0.215      4.85         6.68   2.4% ↓
      8      0.221      4.71         6.31   5.6% ↓
      9      0.227      4.71         6.30   0.2% ↓
     10      0.234      4.70         6.29   0.2% ↓
     11      0.240      4.70         6.28   0.1% ↓
     12      0.247      4.70         6.27   0.1% ↓
     13      0.253      4.70         6.27   0.1% ↓
     14      0.260      4.69         6.26   0.1% ↓
     15      0.266      4.69         6.26   0.1% ↓
     16      0.272      4.69         6.25   0.1% ↓
     17  

View(html=<145,357 characters>, height=376)

In [25]:
project.display.pattern('d20')

View(html=<312,254 characters>, height=480)

In [26]:
project.display.fit.correlations()

,1,2,3,4,5
1 d20.instrument.calib_twotheta_offset,,,,,
2 d20.peak.broad_gauss_u,-0.24,,,,
3 d20.peak.broad_gauss_v,0.32,-0.94,,,
4 d20.peak.broad_gauss_w,-0.29,0.83,-0.92,,
5 d20.peak.asym_fcj_1,0.78,-0.30,0.39,-0.37,


In [27]:
project.display.structure('cosio')

View(html=<342,757 characters>, height=660)

## Report

The HTML report draws the measured, calculated and difference patterns with the reflection
positions, and lists the structures, the experiments and the fit. Open it in a browser.

In [28]:
project.report.save_html('pd-neut-cwl_cosio-d20_2k.html')

Saving report to 'pd-neut-cwl_cosio-d20_2k.html'


## Save the project

In [29]:
project.save_as('pd-neut-cwl_cosio-d20_2k')

Saving project to 'pd-neut-cwl_cosio-d20_2k'
